In [ ]:
import xarray as xr
import os
import numpy as np

In [ ]:
import glob
from matplotlib import pyplot as plt

In [ ]:
files = [f"/work/bg1602/g260094/SNS/RUN00a/outputs_all/out2d_{day}.nc" for day in range(28, 366)]
files = [f for f in files if os.path.exists(f)]
len(files), files[0], files[-1]

In [ ]:
variables = ["windSpeedX", "windSpeedY", "elevation", "depthAverageVelX", "depthAverageVelY", "sigWaveHeight", "peakPeriod", "meanWavePeriod", "TM10",
             "meanWaveNumber", "meanWaveLength", "meanWaveDirection", "meanDirSpreading",
             "SCHISM_hgrid_face_nodes", "SCHISM_hgrid_edge_nodes", "dryFlagNode", "depth",
             "continuousPeakPeriod", "dominantDirection", "rmsOrbitalVelocity", "bottomExcursionPerio", "peakPhaseVel", "peakWaveLength", "dissRateDepBreaking",
             "rollerDissRate",
             "SCHISM_hgrid_node_x", "SCHISM_hgrid_node_y", "SCHISM_hgrid_face_x", "SCHISM_hgrid_face_y", "SCHISM_hgrid_edge_x", "SCHISM_hgrid_edge_y",
            ]

In [ ]:
ds = xr.open_mfdataset(files, chunks="auto")

In [ ]:
ds.time.values[-1]

In [ ]:
ds = ds[variables]
for var in variables:
    ds[var] = ds[var].astype("float32")

for coord in ds.coords:
    if coord != "time":
        ds[coord] = ds[coord].astype("float32")

In [ ]:
encoding = {var: {'chunks': (24, 589428)} for var in ds.data_vars}

In [ ]:
ds

In [ ]:
%%time
ds.to_zarr("/work/ka1176/shared_data/2025-wwm-emulator-schism/processed_SNS_RUN00a_LevanteJP.zarr", mode="w", encoding=encoding)

In [ ]:
zarrds = xr.open_dataset("/work/ka1176/shared_data/2025-wwm-emulator-schism/processed_SNS_RUN00a_LevanteJP.zarr", engine="zarr", chunks={})
zarrds

In [ ]:
zarrds.time.values[[0, -1]]

In [ ]:
# TODO check times

In [ ]:
trainds_winter = zarrds.sel(time=slice("2020-01-01", "2020-05-31"))
trainds_summer = zarrds.sel(time=slice("2020-08-01", "2020-10-31"))

In [ ]:
trainds = xr.concat([trainds_winter, trainds_summer], dim="time")
trainds.time.plot(ls="", marker="x")

In [ ]:
windwavenet_targets = ["sigWaveHeight", "meanWaveDirection", "meanWavePeriod", "peakPeriod",
                       "continuousPeakPeriod", "dominantDirection", "rmsOrbitalVelocity", "bottomExcursionPerio",
                       "peakPhaseVel", "peakWaveLength", "dissRateDepBreaking", "rollerDissRate"]

In [ ]:
fig, ax = plt.subplots(3, 4, figsize=(4*3, 3*3))
ax = ax.flatten()

for i, t in enumerate(windwavenet_targets):
    vals = zarrds[t].isel(time=slice(None, None, 24), nSCHISM_hgrid_node=slice(None, None, 10000)).values.flatten()
    ax[i].hist(vals);
    ax[i].set_title(t);

fig.tight_layout();

In [ ]:
for i, t1 in enumerate(windwavenet_targets):
    for j, t2 in enumerate(windwavenet_targets):
        if j <= i:
            continue
        vals1 = zarrds[t1].isel(time=slice(None, None, 24), nSCHISM_hgrid_node=slice(None, None, 10000)).values.flatten()
        vals2 = zarrds[t2].isel(time=slice(None, None, 24), nSCHISM_hgrid_node=slice(None, None, 10000)).values.flatten()
        print(f"C({t1}, {t2}) = {np.corrcoef(vals1, vals2)[0,1]:.2f}")

Create scaling file that can be used in YAML

In [ ]:
for var in ["windSpeedX", "windSpeedY", "elevation", "depthAverageVelX", "depthAverageVelY", 
            "sigWaveHeight", "peakPeriod", "meanWavePeriod", "meanWaveDirection", "meanDirSpreading", 
            "depth", "continuousPeakPeriod", "dominantDirection", "rmsOrbitalVelocity", 
            "bottomExcursionPerio", "peakPhaseVel", "peakWaveLength", "dissRateDepBreaking",
            "rollerDissRate",
           ]:
    if var in windwavenet_targets:
        pvar = "prev_" + var
    else:
        pvar = var
    print(f"{pvar}:")
    print(f"  mean: {trainds[var].mean().values:.4f}")
    print(f"  std: {trainds[var].std().values:.4f}")

## Statistics

In [ ]:
testds_winter = zarrds.sel(time=slice("2020-07-01", "2020-07-31"))
testds_summer = zarrds.sel(time=slice("2020-12-01", "2020-12-31"))
testds = xr.concat([testds_winter, testds_summer], dim="time")


In [ ]:
testds["meanWavePeriod"].plot.hist();

In [ ]:
np.quantile(testds["meanWavePeriod"], [0.01, 0.1, 0.9, 0.99])

In [ ]:
testds["meanWaveDirection"].plot.hist();

In [ ]:
(testds_winter["meanWaveDirection"] - testds_winter.shift(time=1)["meanWaveDirection"]).plot.hist();